## 🧊 实验 1：FrozenLake 入门

在本实验中，我们将探索 **FrozenLake**，这是由 [Gymnasium](https://gymnasium.farama.org/environments/toy_text/frozen_lake/) 提供的一个经典强化学习环境。  
FrozenLake 是一个简单的 **网格世界（grid world）**，智能体需要从 **起点格子 (S)** 出发，在不掉入 **冰洞 (H)** 的情况下，到达 **目标格子 (G)**。  
在每一步中，智能体可以选择 **上、下、左、右** 移动，但由于冰面可能很滑，所选择的动作 **不一定会朝期望的方向移动**。

- **状态空间（State Space）：** 每个网格单元都是一个离散状态（默认 4×4 = 16 个状态）。  
- **动作空间（Action Space）：** 4 个动作 —— `LEFT (0)`、`DOWN (1)`、`RIGHT (2)`、`UP (3)`。  
- **奖励函数（Reward Function）：** 到达目标时奖励 +1，其余情况奖励为 0。  
- **回合终止（Episode Termination）：** 当智能体掉入冰洞或到达目标时，回合结束。

FrozenLake 是强化学习实验的一个很好的起点，因为它具有以下特点：

- **易于可视化**：有助于理解强化学习的基本交互循环（`reset → step → render`）。
- **规模小且离散**：非常适合测试价值迭代（Value Iteration）、策略迭代（Policy Iteration）以及基础强化学习算法。
- **可定制性强**：可以调整地图大小以及环境随机性（`is_slippery=True/False`）。

在今天的实验中，我们将运行一个 **随机策略（random agent）**，以熟悉 Gymnasium 的 API，并观察环境的行为方式。在此基础上，后续实验将进一步探索更智能的策略。

In [2]:
import gymnasium as gym
import numpy as np
from gymnasium.envs.toy_text.frozen_lake import generate_random_map

### 🔧 设置 FrozenLake（4×4）

我们先创建一个 **4×4 的 FrozenLake 环境**，并将其设置为 **确定性模式**（`is_slippery=False`），这样在执行动作时就不会出现随机滑动，便于逐步观察环境行为。

我们将完成以下几个步骤：

1. 初始化环境（Initialize the environment）。
2. 查看 **状态空间**（离散状态的数量）。
3. 查看 **动作空间**（可执行动作的数量）。
4. 打印 **奖励地图**，以观察目标位置和冰洞的位置。

In [3]:
# 1. 创建 FrozenLake 环境（设置为确定性模式，方便观察行为）
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="ansi")

In [8]:
# 2. 查看状态空间和动作空间
print(f"状态空间: {env.observation_space}  -> 共 {env.observation_space.n} 个状态")
print(f"动作空间: {env.action_space}  -> 共 {env.action_space.n} 个动作 (0=左, 1=下, 2=右, 3=上)")

状态空间: Discrete(16)  -> 共 16 个状态
动作空间: Discrete(4)  -> 共 4 个动作 (0=左, 1=下, 2=右, 3=上)


In [9]:
# 3. 渲染初始状态
obs, info = env.reset(seed=42)
print("\n初始地图：")
print(env.render())


初始地图：

SFFF
FHFH
FFFH
HFFG



In [10]:
# 4. 使用 env.unwrapped.P 查看奖励地图
reward_map = np.zeros(env.observation_space.n)
P = env.unwrapped.P  # 访问底层的 FrozenLakeEnv 转移结构

for state, transitions in P.items():
    for action, outcomes in transitions.items():
        for prob, next_state, reward, done in outcomes:
            reward_map[next_state] = max(reward_map[next_state], reward)

print("奖励地图（reshape 为 4×4）：")
print(reward_map.reshape(4, 4))

奖励地图（reshape 为 4×4）：
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 1.]]


### 🔑 Gym 核心 API：`reset()` 和 `step()`

每一个 Gym 环境都遵循相同的基本交互模式：  
首先通过 **`reset()`** 开始一个新的回合（episode），然后通过 **`step()`** 在环境中不断执行动作，直到该回合结束。

In [ ]:
env.reset()
obs, info = env.reset(seed=0)
print(obs, info)

- **`obs`** → the initial state (an integer for FrozenLake, `0–15` for a 4×4 grid).
- **`info`** → extra diagnostic info (not used much here).

In [ ]:
obs, reward, terminated, truncated, info = env.step(action)
print(action, obs, reward, terminated, truncated, info  )

#### 🟡 `env.step(action)`

| 变量 | 含义 |
|------|------|
| **`action`** | 整数 `0–3`：表示你选择的动作（见下表）。 |
| **`obs`** | **下一状态**（在 4×4 网格中是 `0` 到 `15` 之间的整数）。 |
| **`reward`** | 当前这一步获得的奖励（到达目标为 `+1`，其他情况为 `0`）。 |
| **`terminated`** | 若回合因为 **到达目标** 或 **掉入冰洞** 而结束，则为 `True`。 |
| **`truncated`** | 若回合因为达到 **时间上限** 而结束，则为 `True`（在 FrozenLake 中较少见）。 |
| **`info`** | 额外信息（在 FrozenLake 中通常不需要使用）。 |

#### 🧭 FrozenLake 的动作映射

| 动作编号 | 方向 | 符号 |
|---------|------|------|
| `0` | **LEFT（左）** | ⬅️ |
| `1` | **DOWN（下）** | ⬇️ |
| `2` | **RIGHT（右）** | ➡️ |
| `3` | **UP（上）** | ⬆️ |

---

你需要不断重复调用 `env.step(action)`，直到 **`terminated` 或 `truncated`** 变为 `True`。  
随后调用 **`env.reset()`**，即可开始一个新的回合。

### 🏃 练习：实现 `run_random_trajectory`

请编写一个名为 **`run_random_trajectory`** 的函数，要求如下：

1. 使用 **`reset()`** 重置环境，并获得初始状态。
2. 使用 `env.render()` 打印 **初始网格**。
3. 在最多执行 `max_steps` 步的循环中：
   - 使用 `env.action_space.sample()` 随机采样一个动作；
   - 调用 `env.step(action)`，并将返回值解包为  
     `obs, reward, terminated, truncated, info`；
   - 打印 **当前步数**、所选动作、获得的奖励，以及当前网格；
   - 如果 `terminated` 或 `truncated` 为 `True`，则提前结束循环。
4. 在函数结束时，打印该回合累计获得的 **总奖励（total reward）**。

完成后，请使用不同的随机种子运行一条或多条轨迹。


```python
# 示例用法（实现函数后运行）
for i in range(3):
    print(f"=== Trajectory {i+1} ===")
    run_random_trajectory(env, max_steps=20, seed=100 + i)
```

#### 💡 提示
- 记得在函数开头调用 `env.reset(seed=...)`。
- 可以使用类似 `for step in range(max_steps):` 的循环。
- 使用 f-string 清晰地格式化输出，例如  
  `Step {step+1}: Action={action} -> Reward={reward}`
- 当 `terminated or truncated` 为 `True` 时，应提前停止循环。

> **目标：** 你应该能够看到智能体一步一步地移动，并最终掉进冰洞或到达目标。输出格式应与教师示例大致相似，但不必完全一致。

In [ ]:
# Your turn to work on it